In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/cyrilgarcia/notebooks/tsnn
tsnn module path: /Users/cyrilgarcia/notebooks/tsnn/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers



plt.style.use('ggplot')

In [3]:
from dataclasses import dataclass
from torch import nn
device = 'mps'

In [4]:
from typing import Dict

In [5]:
from tsnn.tstorch import models

In [6]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [7]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [8]:
# We work with the following data.

In [9]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [10]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [11]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [12]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

In [13]:
effects = list(dic_data['correl0.1'].keys())
print(effects)

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']


In [14]:
# We will fix the above dataset for now.

In [15]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [16]:
# Let's list here all the models we wish to test on all the data.

In [17]:
def get_models():
    MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

    MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

    trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                            dropout=0.2, roll_y=True).to(device)
    #Note: using the MLP compression seems very bad..

    trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)

    trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)


    dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

    trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)

    trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)


    dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}
    
    return dic_models, dic_models_rollfalse

In [18]:
# For each model we also need to specify the option we will use to fit them.

In [19]:
dic_data['correl0.1']

{'d_lin': <tsnn.generators.generators.Generator at 0x1664aac80>,
 'd_cond': <tsnn.generators.generators.Generator at 0x1664abd00>,
 'd_shift': <tsnn.generators.generators.Generator at 0x165bebd90>,
 'd_cs': <tsnn.generators.generators.Generator at 0x165beacb0>,
 'd_cs_shift': <tsnn.generators.generators.Generator at 0x1664e0040>,
 'd_all': <tsnn.generators.generators.Generator at 0x1664e0070>}

In [20]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

# Function to create table for an effect

In [25]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [22]:
def run_models(effect1, dic_models, dic_models_rollfalse):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [23]:
dic_models, dic_models_rollfalse = get_models()

table_train_lin0, table_test_lin0 = run_models('d_lin', dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.87it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:13<00:00,  2.96it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.59it/s]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:00<00:00,  3.04s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:02<00:00,  6.15s/it]


In [24]:
display(table_train_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.010,0.009,0.036,0.034,0.038,0.172,0.102
correl0.03,0.043,0.037,0.106,0.108,0.110,0.137,0.112
correl0.05,0.040,0.033,0.155,0.166,0.172,0.202,0.185
correl0.1,0.167,0.089,0.305,0.356,0.333,0.371,0.341
correl0.3,0.300,0.191,0.697,0.791,0.729,0.814,0.753
correl0.5,0.310,0.232,0.853,0.905,0.875,0.945,0.906


In [25]:
display(table_test_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,0.008,0.057,0.027,0.054,0.178,0.104
correl0.03,0.020,0.005,0.130,0.094,0.169,0.144,0.150
correl0.05,0.021,0.011,0.080,0.108,0.228,0.253,0.260
correl0.1,0.144,0.038,0.140,0.435,0.433,0.436,0.464
correl0.3,0.298,0.178,0.523,0.864,0.827,0.852,0.837
correl0.5,0.315,0.252,0.827,0.932,0.925,0.954,0.937


In [26]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [31]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [27]:
dic_models, dic_models_rollfalse = get_models()

table_train_cond0, table_test_cond0 = run_models('d_cond', dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.27it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:12<00:00,  3.14it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.63it/s]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:55<00:00,  2.79s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:49<00:00,  5.49s/it]


In [28]:
display(table_train_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.003,0.002,0.022,0.024,0.023,0.042,0.037
correl0.03,0.013,0.026,0.091,0.090,0.090,0.103,0.100
correl0.05,0.018,0.047,0.155,0.145,0.156,0.181,0.169
correl0.1,0.030,0.083,0.289,0.257,0.291,0.336,0.306
correl0.3,0.079,0.176,0.668,0.609,0.674,0.743,0.703
correl0.5,0.092,0.217,0.820,0.735,0.832,0.892,0.866


In [29]:
display(table_test_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.006,-0.003,0.008,0.007,-0.003,0.040,0.015
correl0.03,0.008,0.001,0.005,-0.006,0.017,0.104,0.087
correl0.05,-0.013,-0.005,-0.009,0.010,0.011,0.151,0.152
correl0.1,-0.008,0.004,-0.002,-0.003,0.008,0.287,0.263
correl0.3,0.022,0.015,-0.005,0.001,0.054,0.664,0.648
correl0.5,-0.003,0.007,0.015,0.031,0.140,0.836,0.821


In [30]:
# Saving the data

table_train_cond0.to_csv('table_train_cond.csv', index=True)
table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [31]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [32]:
dic_models, dic_models_rollfalse = get_models()

table_train_shift0, table_test_shift0 = run_models('d_shift', dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:07<00:00,  5.21it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:13<00:00,  3.06it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:20<00:00,  1.02s/it]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [12:02<00:00, 36.12s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [15:36<00:00, 46.83s/it]


In [33]:
display(table_train_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.012,0.012,0.032,0.030,0.033,0.189,0.121
correl0.03,0.014,0.024,0.091,0.092,0.092,0.106,0.100
correl0.05,0.052,0.032,0.150,0.155,0.152,0.195,0.173
correl0.1,0.150,0.086,0.304,0.360,0.314,0.376,0.335
correl0.3,0.308,0.197,0.688,0.778,0.707,0.797,0.739
correl0.5,0.311,0.232,0.850,0.906,0.862,0.928,0.902


In [34]:
display(table_test_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.007,-0.002,0.053,0.029,0.010,0.181,0.120
correl0.03,0.014,0.009,0.104,0.066,0.028,0.104,0.116
correl0.05,0.038,0.012,0.073,0.090,0.042,0.197,0.161
correl0.1,0.118,0.033,0.141,0.443,0.097,0.388,0.355
correl0.3,0.308,0.186,0.537,0.866,0.479,0.815,0.771
correl0.5,0.312,0.248,0.819,0.933,0.774,0.932,0.910


In [35]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [36]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [37]:
dic_models, dic_models_rollfalse = get_models()

table_train_cs0, table_test_cs0 = run_models('d_cs', dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.06it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:09<00:00,  4.37it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:07<00:00,  2.61it/s]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.38s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:52<00:00,  2.62s/it]


In [38]:
display(table_train_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.031,0.017,0.033,0.036,0.037,0.079,0.072
correl0.03,0.031,0.030,0.100,0.101,0.104,0.127,0.109
correl0.05,0.071,0.052,0.161,0.167,0.176,0.192,0.185
correl0.1,0.156,0.084,0.299,0.328,0.327,0.366,0.322
correl0.3,0.300,0.188,0.686,0.781,0.723,0.824,0.733
correl0.5,0.317,0.244,0.856,0.908,0.880,0.955,0.907


In [39]:
display(table_test_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.013,-0.007,0.040,0.017,0.049,0.054,0.061
correl0.03,0.032,0.013,0.129,0.080,0.155,0.161,0.157
correl0.05,0.049,0.024,0.080,0.101,0.232,0.148,0.200
correl0.1,0.128,0.020,0.112,0.333,0.425,0.379,0.361
correl0.3,0.316,0.181,0.478,0.852,0.814,0.846,0.801
correl0.5,0.315,0.257,0.824,0.932,0.926,0.960,0.932


In [40]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [41]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [42]:
dic_models, dic_models_rollfalse = get_models()

table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift', dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.42it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:08<00:00,  4.74it/s]


Running → correl0.01 | trans_1D_T4


...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.31it/s]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:29<00:00,  1.46s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:54<00:00,  2.72s/it]


In [43]:
display(table_train_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.008,0.003,0.034,0.032,0.032,0.007,0.016
correl0.03,0.033,0.026,0.101,0.098,0.101,0.092,0.096
correl0.05,0.051,0.044,0.154,0.163,0.158,0.142,0.149
correl0.1,0.161,0.085,0.305,0.356,0.311,0.274,0.292
correl0.3,0.303,0.191,0.688,0.760,0.700,0.773,0.663
correl0.5,0.314,0.239,0.855,0.911,0.867,0.928,0.898


In [44]:
display(table_test_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.000,0.010,0.041,0.021,0.007,-0.005,-0.010
correl0.03,0.029,0.011,0.118,0.085,0.030,0.014,-0.003
correl0.05,0.034,0.004,0.078,0.125,0.044,0.017,0.015
correl0.1,0.138,0.047,0.147,0.412,0.087,0.029,0.013
correl0.3,0.300,0.177,0.526,0.853,0.453,0.689,0.134
correl0.5,0.319,0.257,0.825,0.933,0.786,0.925,0.891


In [45]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [46]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [47]:
def run_models_all_effect(noise_level1, dic_models, dic_models_rollfalse):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['correl0.01', 'correl0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

## Run on all noise levels

In [48]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_1, all_effects_test_1 = run_models_all_effect("correl0.01", dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:05<00:00,  7.50it/s]


Running → correl0.01 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:09<00:00,  4.14it/s]


Running → correl0.01 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:16<00:00,  2.39it/s]


Running → correl0.01 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:30<00:00,  1.51s/it]


Running → correl0.01 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [02:47<00:00,  2.79s/it]


In [49]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_3, all_effects_test_3 = run_models_all_effect("correl0.03", dic_models, dic_models_rollfalse)

Running → correl0.03 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:06<00:00,  5.89it/s]


Running → correl0.03 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:09<00:00,  4.25it/s]


Running → correl0.03 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:16<00:00,  2.40it/s]


Running → correl0.03 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [01:00<00:00,  1.51s/it]


Running → correl0.03 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [02:02<00:00,  3.07s/it]


In [50]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_5, all_effects_test_5 = run_models_all_effect("correl0.05", dic_models, dic_models_rollfalse)

Running → correl0.05 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  7.46it/s]


Running → correl0.05 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.38it/s]


Running → correl0.05 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.40it/s]


Running → correl0.05 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:30<00:00,  1.55s/it]


Running → correl0.05 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:55<00:00,  2.76s/it]


In [51]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_10, all_effects_test_10 = run_models_all_effect("correl0.1", dic_models, dic_models_rollfalse)

Running → correl0.1 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  6.90it/s]


Running → correl0.1 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.28it/s]


Running → correl0.1 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.38it/s]


Running → correl0.1 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:29<00:00,  1.48s/it]


Running → correl0.1 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:07<00:00,  3.37s/it]


In [52]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_30, all_effects_test_30 = run_models_all_effect("correl0.3", dic_models, dic_models_rollfalse)

Running → correl0.3 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.48it/s]


Running → correl0.3 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.22it/s]


Running → correl0.3 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:09<00:00,  2.21it/s]


Running → correl0.3 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:47<00:00,  2.36s/it]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:18<00:00,  3.90s/it]


In [53]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_50, all_effects_test_50 = run_models_all_effect("correl0.5", dic_models, dic_models_rollfalse)

Running → correl0.5 | MLP_global


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  7.30it/s]


Running → correl0.5 | MLP_2D


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.46it/s]


Running → correl0.5 | trans_1D_T4


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.28it/s]


Running → correl0.5 | trans_2D_TCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:29<00:00,  1.48s/it]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:56<00:00,  2.81s/it]


In [54]:
display(all_effects_train_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.031,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.013,0.445,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.014,0.455,0.003,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.018,0.456,0.006,0.013,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.012,0.450,0.002,0.001,0.010,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.014,0.447,-0.003,0.002,0.003,0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.112,0.010,-0.017,0.012,0.002,0.015,0.010,nan,nan,nan,nan,nan,nan
boosting,0.263,0.006,-0.005,0.010,0.004,-0.003,0.008,0.316,nan,nan,nan,nan,nan
MLP_global,0.989,0.032,0.015,0.014,0.018,0.010,0.015,0.111,0.263,nan,nan,nan,nan
MLP_2D,0.976,0.031,0.014,0.013,0.018,0.010,0.013,0.108,0.252,0.967,nan,nan,nan


In [55]:
display(all_effects_test_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.018,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.014,0.444,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,-0.001,0.440,-0.013,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.012,0.445,-0.011,-0.003,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.019,0.438,0.005,-0.011,-0.010,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,-0.004,0.443,0.002,-0.001,0.004,-0.022,nan,nan,nan,nan,nan,nan,nan
lasso_full,-0.003,0.015,-0.017,-0.003,0.021,0.021,0.011,nan,nan,nan,nan,nan,nan
boosting,-0.010,-0.003,-0.000,-0.004,-0.002,-0.010,0.010,0.227,nan,nan,nan,nan,nan
MLP_global,0.004,0.035,0.023,-0.018,0.028,0.009,0.036,0.074,0.053,nan,nan,nan,nan
MLP_2D,0.015,0.018,0.004,-0.002,0.002,0.017,0.018,0.034,0.024,0.316,nan,nan,nan


In [56]:
display(all_effects_train_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.092,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.043,0.450,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.046,0.446,-0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.037,0.452,0.013,0.001,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.037,0.446,0.004,-0.002,-0.001,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.044,0.450,0.003,0.008,0.002,-0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.103,0.016,0.006,-0.003,0.024,0.002,0.007,nan,nan,nan,nan,nan,nan
boosting,0.260,0.024,0.010,0.005,0.015,0.002,0.022,0.294,nan,nan,nan,nan,nan
MLP_global,0.989,0.092,0.043,0.046,0.036,0.039,0.043,0.097,0.249,nan,nan,nan,nan
MLP_2D,0.977,0.091,0.043,0.045,0.037,0.037,0.043,0.102,0.253,0.967,nan,nan,nan


In [57]:
display(all_effects_test_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.111,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.054,0.460,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.050,0.443,0.008,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.052,0.455,0.007,0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.045,0.456,0.014,0.006,0.006,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.049,0.431,-0.002,-0.021,0.005,-0.012,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.005,-0.005,-0.010,0.007,0.013,-0.015,-0.005,nan,nan,nan,nan,nan,nan
boosting,-0.012,-0.001,0.010,-0.002,0.003,0.003,-0.016,0.154,nan,nan,nan,nan,nan
MLP_global,0.010,0.107,0.052,0.017,0.058,0.038,0.075,0.057,0.033,nan,nan,nan,nan
MLP_2D,0.002,0.055,0.029,0.015,0.024,0.016,0.040,0.021,0.018,0.307,nan,nan,nan


In [58]:
display(all_effects_train_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.155,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.069,0.454,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.074,0.454,0.013,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.071,0.461,0.008,0.012,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.070,0.452,-0.002,0.005,0.021,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.068,0.448,0.014,0.003,-0.003,0.004,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.101,0.047,0.033,0.011,0.040,0.010,0.013,nan,nan,nan,nan,nan,nan
boosting,0.256,0.035,0.021,0.013,0.010,0.015,0.020,0.274,nan,nan,nan,nan,nan
MLP_global,0.984,0.158,0.071,0.073,0.074,0.073,0.068,0.107,0.259,nan,nan,nan,nan
MLP_2D,0.939,0.163,0.077,0.068,0.076,0.077,0.073,0.103,0.241,0.927,nan,nan,nan


In [59]:
display(all_effects_test_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.156,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.072,0.446,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.066,0.440,-0.009,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.077,0.447,0.004,-0.001,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.057,0.453,0.015,0.005,-0.011,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.076,0.457,0.002,-0.004,0.012,0.006,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.012,0.021,0.016,-0.005,0.028,0.007,0.000,nan,nan,nan,nan,nan,nan
boosting,0.007,0.010,0.001,-0.006,0.008,0.014,0.005,0.108,nan,nan,nan,nan,nan
MLP_global,0.030,0.176,0.100,-0.005,0.089,0.113,0.098,0.082,0.038,nan,nan,nan,nan
MLP_2D,0.017,0.139,0.075,-0.014,0.087,0.091,0.073,0.039,0.007,0.335,nan,nan,nan


In [60]:
display(all_effects_train_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.299,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.138,0.464,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.142,0.470,0.023,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.140,0.463,0.018,0.019,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.138,0.471,0.016,0.034,0.022,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.137,0.460,0.021,0.015,0.016,0.023,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.129,0.127,0.050,0.013,0.074,0.089,0.070,nan,nan,nan,nan,nan,nan
boosting,0.249,0.078,0.038,0.031,0.042,0.038,0.033,0.349,nan,nan,nan,nan,nan
MLP_global,0.984,0.306,0.142,0.140,0.143,0.145,0.142,0.132,0.246,nan,nan,nan,nan
MLP_2D,0.942,0.322,0.153,0.136,0.161,0.153,0.146,0.130,0.234,0.930,nan,nan,nan


In [61]:
display(all_effects_test_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.298,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.136,0.470,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.133,0.454,0.021,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.139,0.472,0.015,0.015,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.136,0.464,0.030,-0.003,0.040,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.151,0.475,0.029,0.027,0.037,0.016,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.026,0.095,0.028,-0.003,0.060,0.065,0.072,nan,nan,nan,nan,nan,nan
boosting,0.006,0.017,0.013,0.002,0.018,0.010,-0.003,0.192,nan,nan,nan,nan,nan
MLP_global,0.114,0.343,0.200,0.011,0.198,0.193,0.199,0.126,0.042,nan,nan,nan,nan
MLP_2D,0.110,0.353,0.188,0.032,0.224,0.189,0.193,0.087,0.027,0.427,nan,nan,nan


In [62]:
display(all_effects_train_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.690,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.401,0.572,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.390,0.569,0.164,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.388,0.560,0.152,0.142,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.383,0.565,0.150,0.156,0.143,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.395,0.571,0.159,0.150,0.153,0.153,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.210,0.283,0.189,0.082,0.159,0.174,0.200,nan,nan,nan,nan,nan,nan
boosting,0.259,0.183,0.113,0.093,0.098,0.105,0.109,0.589,nan,nan,nan,nan,nan
MLP_global,0.987,0.703,0.411,0.387,0.397,0.394,0.405,0.217,0.259,nan,nan,nan,nan
MLP_2D,0.955,0.729,0.439,0.377,0.419,0.412,0.423,0.221,0.247,0.948,nan,nan,nan


In [63]:
display(all_effects_test_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.688,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.398,0.562,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.392,0.569,0.159,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.377,0.561,0.142,0.142,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.391,0.572,0.144,0.161,0.153,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.398,0.575,0.158,0.150,0.154,0.167,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.197,0.279,0.197,0.077,0.156,0.161,0.200,nan,nan,nan,nan,nan,nan
boosting,0.112,0.153,0.119,0.040,0.085,0.096,0.096,0.568,nan,nan,nan,nan,nan
MLP_global,0.503,0.735,0.473,0.205,0.464,0.465,0.481,0.251,0.144,nan,nan,nan,nan
MLP_2D,0.542,0.779,0.508,0.230,0.501,0.483,0.491,0.245,0.143,0.730,nan,nan,nan


In [64]:
display(all_effects_train_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.844,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.574,0.682,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.574,0.679,0.332,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.576,0.682,0.331,0.332,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.574,0.685,0.333,0.330,0.335,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.580,0.683,0.333,0.327,0.329,0.338,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.260,0.305,0.224,0.156,0.216,0.215,0.231,nan,nan,nan,nan,nan,nan
boosting,0.265,0.235,0.162,0.151,0.162,0.158,0.169,0.739,nan,nan,nan,nan,nan
MLP_global,0.989,0.852,0.582,0.568,0.583,0.583,0.588,0.269,0.269,nan,nan,nan,nan
MLP_2D,0.963,0.882,0.605,0.559,0.612,0.613,0.618,0.268,0.252,0.960,nan,nan,nan


In [65]:
display(all_effects_test_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.843,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.578,0.684,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.573,0.676,0.332,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.577,0.683,0.338,0.321,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.573,0.683,0.331,0.334,0.326,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.569,0.678,0.325,0.318,0.337,0.332,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.256,0.303,0.220,0.147,0.218,0.217,0.227,nan,nan,nan,nan,nan,nan
boosting,0.197,0.235,0.160,0.113,0.165,0.173,0.189,0.786,nan,nan,nan,nan,nan
MLP_global,0.728,0.865,0.639,0.409,0.635,0.625,0.637,0.285,0.231,nan,nan,nan,nan
MLP_2D,0.750,0.894,0.645,0.448,0.651,0.647,0.650,0.280,0.223,0.865,nan,nan,nan


In [66]:
# Let's also save all the tables above.
all_effects_train_1.to_csv('all_effects_train_1.csv', index=True)
all_effects_test_1.to_csv('all_effects_test_1.csv', index=True)

all_effects_train_3.to_csv('all_effects_train_3.csv', index=True)
all_effects_test_3.to_csv('all_effects_test_3.csv', index=True)

all_effects_train_5.to_csv('all_effects_train_5.csv', index=True)
all_effects_test_5.to_csv('all_effects_test_5.csv', index=True)

all_effects_train_10.to_csv('all_effects_train_10.csv', index=True)
all_effects_test_10.to_csv('all_effects_test_10.csv', index=True)

all_effects_train_30.to_csv('all_effects_train_30.csv', index=True)
all_effects_test_30.to_csv('all_effects_test_30.csv', index=True)

all_effects_train_50.to_csv('all_effects_train_50.csv', index=True)
all_effects_test_50.to_csv('all_effects_test_50.csv', index=True)

# Testing sparsity

In [50]:
# Let's now test sparsity at different levels of noise for the different effects.